# Use Python API to automate AutoAI deployment lifecycle

This notebook contains the steps and code to demonstrate support of AI Lifecycle features of the AutoAI model in watsonx.ai service. It contains steps and code to work with [ibm-watsonx-ai](https://pypi.python.org/pypi/ibm-watsonx-ai) SDK available in PyPI repository. It also introduces commands for training, persisting and deploying model, scoring it, updating the model and redeploying it.

Some familiarity with Python is helpful. This notebook uses Python 3.12.


## Learning goals

The learning goals of this notebook are:

-  List all deprecated and unsupported deployments.
-  Identify AutoAI models that need to be retrained.
-  Work with watsonx.ai experiments to re-train AutoAI models.
-  Persist an updated AutoAI model in watsonx.ai repository.
-  Redeploy model in-place.
-  Score sample records using client library.


## Contents

This notebook contains the following parts:

1. [Set up the environment](#1.-Set-up-the-environment)
2. [Deployments state check](#2.-Deployments-state-check)
3. [Identification of model requiring retraining](#3.-Identification-of-model-requiring-retraining)
4. [Experiment re-run](#4.-Experiment-re-run)
5. [Store the model in repository](#5.-Store-the-model-in-repository)
6. [Redeploy and score new version of the model](#6.-Redeploy-and-score-new-version-of-the-model)
7. [Cleanup](#7.-Cleanup)
8. [Summary and next steps](#8.-Summary-and-next-steps)

<a id="1.-Set-up-the-environment"></a>
## 1. Set up the environment

Before you use the sample code in this notebook, contact with your IBM Cloud Pak® for Data administrator and ask for your account credentials.

### Install dependencies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install -U wget | tail -n 1
%pip install "scikit-learn==1.6.1" | tail -n 1
%pip install -U autoai-libs | tail -n 1
%pip install -U ibm-watsonx-ai | tail -n 1

#### Define credentials

Authenticate the watsonx.ai Runtime service on IBM Cloud Pak® for Data. You need to provide the **admin's** `username` and the platform `url`.

In [2]:
username = "PASTE YOUR USERNAME HERE"
url = "PASTE THE PLATFORM URL HERE"

Use the **admin's** `api_key` to authenticate watsonx.ai Runtime services:

In [ ]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    username=username,
    api_key=getpass.getpass("Enter your watsonx.ai API key and hit enter: "),
    url=url,
    instance_id="openshift",
    version="5.4",
)

Alternatively you can use the **admin's** `password`:

In [3]:
import getpass

from ibm_watsonx_ai import Credentials

if "credentials" not in locals() or not credentials.api_key:
    credentials = Credentials(
        username=username,
        password=getpass.getpass("Enter your watsonx.ai password and hit enter: "),
        url=url,
        instance_id="openshift",
        version="5.4",
    )

#### Create `APIClient` instance

In [4]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials)

### Working with spaces

First of all, you need to create a space that will be used for your work. If you do not have space already created, you can use `{PLATFORM_URL}/ml-runtime/spaces?context=icp4data` to create one.

- Click New Deployment Space
- Create an empty space
- Go to space `Settings` tab
- Copy `space_id` and paste it below

**Tip**: You can also use SDK to prepare the space for your work. More information can be found [here](https://github.com/IBM/watsonx-ai-samples/blob/master/cpd5.4/notebooks/python_sdk/instance-management/Space%20management.ipynb).


You can use the `list` method to print all existing spaces.

In [ ]:
client.spaces.list(limit=10)

Extract all space IDs

In [5]:
space_ids = [
    space["metadata"]["id"] for space in client.spaces.get_details()["resources"]
]

space_ids[:5]

['e63c09cd-8d56-4c90-91ab-1a634c2e1988',
 '892f33cd-5522-4944-83c5-9505b801df9b',
 '1e7ec088-6732-478b-a66b-c6f060ec68f0',
 '364bb957-556d-47e1-8cd2-09f17db1d845']

<a id="2.-Deployments-state-check"></a>
## 2. Deployments state check
Iterate over spaces and search for `deprecated` and `unsupported` deployments. Next, identify models requiring re-training.

In [6]:
from ibm_watsonx_ai.lifecycle import SpecStates

for space_id in space_ids[:5]:
    client.set.default_space(space_id)
    print(f"****** SPACE {space_id} ******")
    print(client.deployments.get_details(spec_state=SpecStates.DEPRECATED))
    print(client.deployments.get_details(spec_state=SpecStates.UNSUPPORTED))

****** SPACE e63c09cd-8d56-4c90-91ab-1a634c2e1988 ******
{'resources': []}
{'resources': []}
****** SPACE 892f33cd-5522-4944-83c5-9505b801df9b ******
{'resources': []}
{'resources': []}
****** SPACE 1e7ec088-6732-478b-a66b-c6f060ec68f0 ******
{'resources': []}
{'resources': []}
****** SPACE 364bb957-556d-47e1-8cd2-09f17db1d845 ******
{'resources': []}
{'resources': []}


You can also list deployments under particular space. The output contains `SPEC_STATE` and `SPEC_REPLACEMENT`. Set the working space.

In [7]:
deployment_space_id = "PASTE YOUR SPACE ID HERE"
client.set.default_space(deployment_space_id)

'SUCCESS'

List deployments under this space.

In [8]:
client.deployments.list()

,ID,NAME,STATE,CREATED,ARTIFACT_TYPE,SPEC_STATE,SPEC_REPLACEMENT
0,26de40f2-67e6-44a3-af5d-2005c78ec552,AutoAI credit-risk deployment,ready,2026-04-24T10:12:46.282Z,model,supported,


<a id="3.-Identification-of-model-requiring-retraining"></a>
## 3. Identification of model requiring retraining
Pick up deployment of the AutoAI model you wish to retrain. 

**Hint**: You can also do that programatically in the loop sequence over spaces check (`Check the state of your deployments` cell).
**Hint**: You can also use software_specification information (model details) to identify models and deployments that are not yet deprecated but can be retrained (updated software specification is available).

In [9]:
deployment_id = "PASTE YOUR DEPLOYMENT ID HERE"

deployment_details = client.deployments.get_details(deployment_id)
deployed_model_id = deployment_details["entity"]["asset"]["id"]

deployed_model_id

Note: online_url is deprecated and will be removed in a future release. Use serving_urls instead.


'005d307c-bcf6-4a7e-a5fe-fd5f202d4383'

#### Extract the deployed model's details (including the pipeline information).

In [10]:
import json

deployed_model_details = client.repository.get_model_details(deployed_model_id)
deployed_pipeline_id = deployed_model_details["entity"]["pipeline"]["id"]

deployed_pipeline_details = client.repository.get_details(deployed_pipeline_id)
experiment_params = deployed_pipeline_details["entity"]["document"]["pipelines"][0][
    "nodes"
][0]["parameters"]

optimization_params = experiment_params["optimization"]

print("Experiment parameters:", json.dumps(experiment_params, indent=2))
print("Optimization parameters:", json.dumps(optimization_params, indent=2))

Experiment parameters: {
  "drop_duplicates": true,
  "encoding": "utf-8",
  "input_file_separator": ",",
  "optimization": {
    "compute_pipeline_notebooks_flag": true,
    "label": "Risk",
    "learning_type": "binary",
    "retrain_on_holdout": true,
    "run_cognito_flag": true,
    "scorer_for_ranking": "roc_auc"
  },
  "output_logs": true,
  "stage_flag": true
}
Optimization parameters: {
  "compute_pipeline_notebooks_flag": true,
  "label": "Risk",
  "learning_type": "binary",
  "retrain_on_holdout": true,
  "run_cognito_flag": true,
  "scorer_for_ranking": "roc_auc"
}


#### Find the AutoAI experiment runs matching the extracted pipeline

Extract the project ID where the training took place.

**Tip:** For more information about using AutoAI with projects, see [this sample notebook](https://github.com/IBM/watsonx-ai-samples/blob/master/cpd5.4/notebooks/python_sdk/experiments/autoai/Use%20AutoAI%20with%20Watson%20Studio%20project.ipynb).

**Note:** If the training took place in a space, please update accordingly.

In [11]:
try:
    training_project_id = deployed_pipeline_details["metadata"]["tags"][0].split(".")[1]
except LookupError:
    training_project_id = input("Please enter your project_id (hit enter): ")

#### Extract AutoAI experiment `training_id`

The `training_id` is available in model's details.

In [12]:
run_id = deployed_model_details["entity"]["training_id"]
print("AutoAI experiment training_id found in model details:", run_id)

AutoAI experiment training_id found in model details: 3d5db948-8566-4424-869c-dcaec7bf036f


<a id="4.-Experiment-re-run"></a>
## 4. Experiment re-run

Set the training `project_id` (where data asset resides) to retrain AutoAI models.

In [13]:
from ibm_watsonx_ai.experiment import AutoAI

experiment = AutoAI(credentials, project_id=training_project_id)
optimizer = experiment.runs.get_optimizer(run_id=run_id)

In [14]:
from ibm_watsonx_ai.utils.autoai.errors import TestDataNotPresent

training_data_reference = optimizer.get_data_connections()
try:
    test_data_reference = optimizer.get_test_data_connections()
except TestDataNotPresent:
    test_data_reference = None

User defined (test / holdout) data is not present for this AutoAI experiment.
Reason: User specified test data was not present in this experiment. Try to use 'with_holdout_split' parameter for original training_data_references to retrieve test data.


In [15]:
train_details = optimizer.fit(
    training_data_references=training_data_reference,
    test_data_references=test_data_reference,
)

Training job c9570b9e-14eb-4b19-8aeb-b0c669b60999 completed: 100%|████████| [01:43<00:00,  1.04s/it]


### Explore experiment's results
Connect to finished experiment and preview the results.

In [16]:
optimizer.summary()

,Enhancements,Estimator,training_roc_auc_(optimized),holdout_average_precision,holdout_log_loss,training_accuracy,holdout_roc_auc,training_balanced_accuracy,training_f1,holdout_precision,training_average_precision,training_log_loss,holdout_recall,training_precision,holdout_accuracy,holdout_balanced_accuracy,training_recall,holdout_f1
Pipeline Name,,,,,,,,,,,,,,,,,,
Pipeline_7,"HPO, FE",SnapSVMClassifier,0.853020,0.525432,3.225722,0.772432,0.823529,0.774273,0.824698,0.846154,0.932992,NaN,0.647059,0.889431,0.68,0.698529,0.769231,0.733333
Pipeline_1,,SnapLogisticRegression,0.827303,0.517600,0.515150,0.718799,0.808824,0.752635,0.767440,0.916667,0.920583,0.553255,0.647059,0.904184,0.72,0.761029,0.666667,0.758621
Pipeline_6,HPO,SnapSVMClassifier,0.832611,0.527391,3.142090,0.745586,0.786765,0.763454,0.797148,0.857143,0.924090,NaN,0.705882,0.896051,0.72,0.727941,0.717949,0.774194
Pipeline_3,"HPO, FE",SnapLogisticRegression,0.852273,0.536944,0.603480,0.736697,0.786765,0.761085,0.787066,0.846154,0.924688,0.565221,0.647059,0.901930,0.68,0.698529,0.698718,0.733333
Pipeline_8,"HPO, FE, HPO",SnapSVMClassifier,0.860558,0.543874,3.158971,0.772372,0.772059,0.778314,0.823502,0.857143,0.934611,NaN,0.705882,0.895058,0.72,0.727941,0.762821,0.774194
Pipeline_2,HPO,SnapLogisticRegression,0.850943,0.539802,0.587449,0.732132,0.772059,0.762250,0.780615,0.833333,0.930346,0.587196,0.588235,0.906850,0.64,0.669118,0.685897,0.689655
Pipeline_4,"HPO, FE, HPO",SnapLogisticRegression,0.852843,0.544014,0.650271,0.736697,0.764706,0.761085,0.787066,0.846154,0.925940,0.649749,0.647059,0.901930,0.68,0.698529,0.698718,0.733333
Pipeline_5,,SnapSVMClassifier,0.827126,0.511277,1.772579,0.749910,0.764706,0.741423,0.808763,0.923077,0.924663,NaN,0.705882,0.862926,0.76,0.790441,0.762821,0.800000


### Evaluate the best model locally

Load the model for test purposes.

**Hint:** The best model is returned automatically if no `pipeline_name` provided.

In [17]:
pipeline_name = "Pipeline_4"
pipeline_model = optimizer.get_pipeline(pipeline_name=pipeline_name, astype="sklearn")
pipeline_model

Pipeline(steps=[('featureunion',
                 FeatureUnion(transformer_list=[('float32_transform_139980656910528',
                                                 Pipeline(steps=[('numpycolumnselector',
                                                                  NumpyColumnSelector(columns=[0,
                                                                                               2,
                                                                                               3,
                                                                                               5,
                                                                                               6,
                                                                                               7,
                                                                                               8,
                                                                                               9,
                                                                                               10,
                                                                                               11,
                                                                                               13,
                                                                                               14,
                                                                                               15,
                                                                                               16,
                                                                                               17,
                                                                                               18,
                                                                                               19])),
                                                                 ('compressstrings',
                                                                  CompressStrings(compress_type='hash',
                                                                                  dtypes_list=['char_str',
                                                                                               'char_str',
                                                                                               'char_str',
                                                                                               'char_str',
                                                                                               'char_str',
                                                                                               'int_num',
                                                                                               'c...
                 autoai_libs.cognito.transforms.transform_utils.FS1(cols_ids_must_keep = range(0, 20), additional_col_count_to_keep = 20, ptype = 'classification')),
                ('snaplogisticregression',
                 SnapLogisticRegression(class_weight='balanced',
                                        device_ids=array([], dtype=uint32),
                                        dual=False, fit_intercept=True,
                                        grad_clip=1.0, max_iter=269,
                                        normalize=True, privacy_epsilon=10.0,
                                        random_state=33,
                                        regularizer=99.62154635907312))])

This cell constructs the cell scorer based on the experiment metadata.

In [18]:
from sklearn.metrics import get_scorer

scorer = get_scorer(optimization_params["scorer_for_ranking"])

#### Read the train and holdout data.

**Hint:** You can also use external test dataset.

In [19]:
connection = optimizer.get_data_connections()[0]
train_X, test_X, train_y, test_y = connection.read(with_holdout_split=True)

#### Calculate the score

In [20]:
score = scorer(pipeline_model, test_X.values, test_y.values)
print(score)

0.7720588235294117


<a id="5.-Store-the-model-in-repository"></a>
## 5. Store the model in repository

Provide `pipeline_name` and `training_id`.

In [21]:
client.set.default_project(training_project_id)

Unsetting the space_id ...


'SUCCESS'

In [22]:
model_metadata = {
    client.repository.ModelMetaNames.NAME: "{0} - {1} - {2}".format(
        deployed_pipeline_details["metadata"]["name"],
        pipeline_name,
        pipeline_model.get_params()["steps"][-1][0],
    )
}
published_model = client.repository.store_model(
    model=pipeline_name,
    meta_props=model_metadata,
    training_id=train_details["metadata"]["id"],
)
updated_model_id = client.repository.get_model_id(published_model)
print("Re-trained model id", updated_model_id)

Re-trained model id b9ac3db5-90e5-454a-9c21-9b869ddbabc0


List stored models.

In [23]:
client.repository.list_models()

,ID,NAME,CREATED,TYPE,SPEC_STATE,SPEC_REPLACEMENT
0,b9ac3db5-90e5-454a-9c21-9b869ddbabc0,Credit Risk Prediction - AutoAI - Pipeline_4 -...,2026-04-24T11:14:50Z,wml-hybrid_0.1,supported,
1,005d307c-bcf6-4a7e-a5fe-fd5f202d4383,AutoAI credit-risk updated model,2026-04-24T10:12:43Z,wml-hybrid_0.1,supported,


<a id="6.-Redeploy-and-score-new-version-of-the-model"></a>
## 6. Redeploy and score new version of the model

In this section, you'll learn how to redeploy new version of the model by using the watsonx.ai Client.

**Hint:** As a best practice please consider using the test space before moving to production.

```
promote(asset_id: str, source_project_id: str, target_space_id: str, rev_id: str = None)
```

### Promote model to deployment space

In [24]:
promoted_model_id = client.spaces.promote(
    asset_id=updated_model_id,
    source_project_id=training_project_id,
    target_space_id=deployment_space_id,
)

Check current deployment details before update.

In [25]:
client.set.default_space(deployment_space_id)
print(json.dumps(client.deployments.get_details(deployment_id), indent=2))

### Update the deployment with new model
**Note:** The update is asynchronous.

In [26]:
metadata = {
    client.deployments.ConfigurationMetaNames.ASSET: {
        "id": promoted_model_id,
    }
}

updated_deployment = client.deployments.update(deployment_id, changes=metadata)

Since ASSET is patched, deployment need to be restarted.


########################################################################

Deployment update for id: '26de40f2-67e6-44a3-af5d-2005c78ec552' started

########################################################################


updating.....
ready


---------------------------------------------------------------------------------------------
Successfully finished deployment update, deployment_id='26de40f2-67e6-44a3-af5d-2005c78ec552'
---------------------------------------------------------------------------------------------




Wait for the deployment update: 

In [27]:
import time

status = None
while status not in ("ready", "failed"):
    time.sleep(2)
    deployment_details = client.deployments.get_details(deployment_id)
    status = deployment_details["entity"]["status"].get("state")
    print(".", status, end=" ")

print("\nDeployment update finished with status: ", status)

Note: online_url is deprecated and will be removed in a future release. Use serving_urls instead.
. ready 
Deployment update finished with status:  ready


#### Get updated deployment details

In [28]:
print(json.dumps(client.deployments.get_details(deployment_id), indent=2))

### Score updated model
Create sample payload and score the deployed model.

In [29]:
scoring_payload = {"input_data": [{"values": test_X[:3]}]}

Use client.deployments.score() method to run scoring.

In [30]:
predictions = client.deployments.score(deployment_id, scoring_payload)

In [31]:
print(json.dumps(predictions, indent=2))

{
  "predictions": [
    {
      "fields": [
        "prediction",
        "probability"
      ],
      "values": [
        [
          "Risk",
          [
            0.417584272051572,
            0.582415727948428
          ]
        ],
        [
          "Risk",
          [
            0.4130392182619398,
            0.5869607817380602
          ]
        ],
        [
          "Risk",
          [
            0.4430112060791491,
            0.5569887939208509
          ]
        ]
      ]
    }
  ]
}


<a id="7.-Cleanup"></a>
## 7. Cleanup

If you want to clean up all created assets:
- experiments
- trainings
- pipelines
- models
- deployments

please follow up this sample [notebook](https://github.com/IBM/watsonx-ai-samples/blob/master/cpd5.4/notebooks/python_sdk/instance-management/Machine%20Learning%20artifacts%20management.ipynb).

<a id="8.-Summary-and-next-steps"></a>
## 8. Summary and next steps

You successfully completed this notebook! You learned how to use scikit-learn machine learning as well as watsonx.ai for model creation and deployment.

Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors and Maintainers

**Lukasz Cmielowski (Former)**, PhD, Senior Technical Staff Member at IBM watsonx.ai

**Dorota Lączak (Former)**, Software Engineer at IBM watsonx.ai

**Rafał Chrzanowski**, Software Engineer at IBM watsonx.ai

**Karol Zmorski**, Software Engineer at IBM watsonx.ai

Copyright © 2023-2026 IBM. This notebook and its source code are released under the terms of the MIT License.